# 04 · Merchant Agent

The seller-side analyst. It owns two graph states:

- **SEARCHING**: finds listings for the request in the pool (fuzzy card name, set, variant,
  minimum condition, excluded variants). Coming back from EVALUATING is the one retry: the
  name match loosens and the set, variant and condition filters drop. With
  `merchant.live: true` it also scrapes Marketplace for the card first.
- **EVALUATING** (external reflection): the risk score from `tools/risk.py` for every
  candidate, then one LLM call that explains each flagged listing in a sentence, grounded in
  the red-flag entries retrieved from Chroma. For a consultative question it also builds the
  market facts (price check).

It plugs into Pablo's graph through the same node signature as his stubs, and fills
`candidate_listings`, `evaluated_listings` (market price, risk, reasons, warning),
`reflection_complete` and `price_check`.

In [ ]:
#| default_exp agents.merchant

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

In [ ]:
#| export
"""Merchant Agent: finds matching listings in the pool (or live on Marketplace) and runs
external reflection on each one: a price check against TCGdex market prices and a red-flag
check against the knowledge base (Section 3, SEARCHING and EVALUATING)."""

import json
import re
import time
from collections.abc import Callable
from typing import Any

from pydantic import BaseModel
from rapidfuzz import fuzz

from src.config import load_config, repo_path
from src.llm import LLM, LLMError, make_llm, offline_handler
from src.pipeline import (
    MarketData,
    MarketplaceScraper,
    PriceBook,
    anonymize,
    apply_scope_filter,
    between,
    build_listings,
    extract_cards,
    load_data,
)
from src.schemas import CONDITION_ORDER, CardListing, EvaluatedListing, ParsedQuery
from src.state import AgentState
from src.tools.prices import condition_adjusted, find_printing, pick_variant_price
from src.tools.risk import RedFlag, RiskResult, assess

## Search

In [ ]:
#| export
def _tokens(s: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", s.lower())

In [ ]:
#| export
def name_matches(query_name: str, listing_name: str, threshold: int) -> bool:
    """Every token of the query must fuzzily match a token of the listing's card name.

    "charizard" matches "Charizard ex", but "pikachu vmax" does not match "Pikachu".
    """
    if not listing_name or listing_name == "unknown":
        return False
    lt = _tokens(listing_name)
    return all(any(fuzz.ratio(q, t) >= threshold for t in lt) for q in _tokens(query_name))

In [ ]:
#| export
def set_matches(query_set: str, listing: CardListing) -> bool:
    """The query's set may be a TCGdex id ("base1") or a name ("Base Set")."""
    if listing.set_code != "unknown" and query_set.lower() == listing.set_code.lower():
        return True
    return bool(listing.set_name) and fuzz.token_set_ratio(query_set, listing.set_name) >= 80

In [ ]:
#| export
def meets_condition(cond: str, minimum: str | None) -> bool:
    if minimum is None:
        return True
    if cond == "unknown":
        return False
    return CONDITION_ORDER.index(cond) >= CONDITION_ORDER.index(minimum)

In [ ]:
#| export
def find(
    listings: list[CardListing], q: ParsedQuery, config: dict[str, Any], relaxed: bool = False
) -> list[CardListing]:
    """Listings matching the request. Relaxed (the one retry) loosens the name match and
    drops the set, variant and condition filters; excluded variants always stay excluded."""
    s = config["search"]
    threshold = s["relaxed_fuzzy_threshold"] if relaxed else s["fuzzy_threshold"]
    out = []
    for item in listings:
        if not q.card_name or not name_matches(q.card_name, item.card_name, threshold):
            continue
        if not relaxed:
            if q.set_code and not q.any_printing and not set_matches(q.set_code, item):
                continue
            if q.variant and item.variant != "unknown" and q.variant != item.variant:
                continue
            if not meets_condition(item.condition, q.min_condition):
                continue
        if item.variant in q.excluded_variants:
            continue
        out.append(item)
    return out[: s["max_candidates"]]

In [ ]:
#| export
def distinct_printings(
    listings: list[CardListing], card_name: str, config: dict[str, Any]
) -> list[str]:
    """Distinct printings among listings for this card name, e.g. ["Charizard from Base Set",
    "Charizard from 151"]. The Platform Agent asks which one when there are several."""
    seen: dict[str, None] = {}
    threshold = config["search"]["fuzzy_threshold"]
    for item in listings:
        if name_matches(card_name or "", item.card_name, threshold):
            seen.setdefault(f"{item.card_name} from {item.set_name or 'an unknown set'}", None)
    return list(seen)

## Reflection: red-flag retrieval and explanations

In [ ]:
#| export
class RedFlagIndex:
    """Chroma retrieval over the red-flag entries, so flag explanations can cite the most
    relevant ones. Keyword matches always come first; Chroma adds related entries. Built on
    first use, and skipped (keywords only) if Chroma or its embedding model is unavailable."""

    def __init__(self, red_flags: list[RedFlag], config: dict[str, Any]):
        self.flags = {f.id: f for f in red_flags}
        self.cfg = config["rag"]
        self._collection = None
        self._tried = False

    def _ensure(self):
        if self._tried or not self.cfg["enabled"]:
            return self._collection
        self._tried = True
        try:
            import chromadb

            client = chromadb.PersistentClient(path=str(repo_path(self.cfg["persist_dir"])))
            col = client.get_or_create_collection("red_flags")
            if col.count() != len(self.flags):
                client.delete_collection("red_flags")
                col = client.get_or_create_collection("red_flags")
                col.add(
                    ids=list(self.flags),
                    documents=[f.as_text() for f in self.flags.values()],
                    metadatas=[{"severity": f.severity} for f in self.flags.values()],
                )
            self._collection = col
        except Exception as e:  # noqa: BLE001 - fall back to keyword matches only
            print(f"[rag] Chroma unavailable, using keyword matches only ({e})")
        return self._collection

    def retrieve(self, text: str, keyword_ids: list[str]) -> list[RedFlag]:
        top_k = self.cfg["top_k"]
        found = [self.flags[i] for i in keyword_ids if i in self.flags]
        col = self._ensure() if len(found) < top_k else None
        if col is not None:
            res = col.query(query_texts=[text], n_results=min(top_k, len(self.flags)))
            for fid in res["ids"][0]:
                if fid in self.flags and self.flags[fid] not in found:
                    found.append(self.flags[fid])
        return found[:top_k]

In [ ]:
#| export
def _sentence(text: str) -> str:
    return (text[:1].upper() + text[1:]).rstrip(".") + "." if text else ""

In [ ]:
#| export
class FlagReason(BaseModel):
    listing_id: str
    reason: str

In [ ]:
#| export
class FlagReasons(BaseModel):
    reasons: list[FlagReason]

In [ ]:
#| export
FLAG_SYSTEM = """You explain to a card buyer why a marketplace listing looks risky.
For each listing, write one short, plain sentence based only on the signals and red-flag
entries given. Do not include prices or other numbers. Do not accuse the seller of fraud;
say what looks wrong and why it matters."""

In [ ]:
#| export
def explain_flags(
    llm: LLM, flagged: list[tuple[CardListing, RiskResult]], index: RedFlagIndex | None
) -> dict[str, str]:
    """One LLM call for all flagged listings: listing_id -> one-sentence reason."""
    if not flagged:
        return {}
    items = []
    for listing, risk in flagged:
        text = f"{listing.raw_title} {listing.raw_description}"
        entries = index.retrieve(text, risk.matched_flags) if index else []
        items.append({
            "listing_id": listing.listing_id,
            "title": listing.raw_title,
            "description": listing.raw_description,
            "signals": risk.signals,
            "red_flag_entries": [{"title": e.title, "why": e.why} for e in entries],
        })  # fmt: skip
    prompt = "Listings to explain (JSON):\n<items>\n" + json.dumps(items, indent=1) + "\n</items>"
    try:
        out = llm.json(tag="flag_reasons", system=FLAG_SYSTEM, prompt=prompt, schema=FlagReasons)
        by_id = {r.listing_id: r.reason for r in out.reasons}
    except LLMError as e:
        print(f"[merchant] flag explanations failed, using the raw signals ({e})")
        by_id = {}
    return {
        listing.listing_id: by_id.get(listing.listing_id) or _sentence("; ".join(risk.signals[:2]))
        for listing, risk in flagged
    }

## Price check

In [ ]:
#| export
def price_check_facts(q: ParsedQuery, data: MarketData, config: dict[str, Any]) -> dict[str, Any]:
    """Market facts for a consultative question like "is $40 fair for a Base Set Charizard?"."""
    entry = find_printing(data.prices, q.card_name or "", q.set_code)
    if not entry or not entry.get("variants"):
        return {"found": False, "card": q.card_name}
    market = pick_variant_price(entry["variants"], q.variant)
    cond = q.min_condition or "LP"
    facts: dict[str, Any] = {
        "found": True,
        "card": entry["name"],
        "set": entry.get("set_name"),
        "condition": cond,
        "condition_assumed": q.min_condition is None,
        "market_price": market,
        "market_for_condition": condition_adjusted(market, cond, config),
        "sample_prices": data.source == "samples",
    }
    if q.asked_price is not None:
        th = config["thresholds"]
        ratio = q.asked_price / facts["market_for_condition"]
        if ratio < th["price_risk_med_ratio"]:
            verdict = "suspiciously low"
        elif ratio <= th["price_check_fair_max"]:
            verdict = "fair"
        else:
            verdict = "on the high side"
        facts.update({"asked_price": q.asked_price, "verdict": verdict})
    return facts

## Live search

Scrape Marketplace for the requested card, then identify, filter and price the new listings with the pipeline functions.

In [ ]:
#| export
def live_term(q: ParsedQuery) -> str:
    """Marketplace search text for a query, e.g. "pokemon charizard base set"."""
    parts = ["pokemon", q.card_name or ""]
    if q.set_code and not q.any_printing:
        parts.append(q.set_code)
    return " ".join(p for p in parts if p).lower()

In [ ]:
#| export
def live_refresh(
    data: MarketData,
    q: ParsedQuery,
    config: dict[str, Any],
    llm: LLM,
    scraper: MarketplaceScraper,
    pricebook: PriceBook,
    log=None,
) -> dict[str, Any]:
    """Scrape Marketplace for this card, identify and price the new listings, add them to
    the pool and to data/live_listings.json. Returns a short summary."""
    m = config["merchant"]
    term = live_term(q)
    items = scraper.search(
        [term], limit=m["live_limit"], max_charge_usd=m["live_max_charge_usd"], save_as="live"
    )
    known = {x.listing_id for x in data.listings}
    rows = [r for r in (anonymize(i) for i in items) if r and r["listing_id"] not in known]
    summary: dict[str, Any] = {"term": term, "scraped": len(items), "added": 0}
    if not rows:
        return summary
    extracted = extract_cards(llm, rows, config["ingest"]["batch_size"], log=log)
    new, dropped = apply_scope_filter(build_listings(rows, extracted))
    summary["dropped"] = dropped
    summary["unresolved_sets"] = pricebook.resolve_sets(new)
    summary["unpriced"] = pricebook.ensure_prices(new)
    pricebook.save()
    data.listings.extend(new)
    data.prices.update({k: v for k, v in pricebook.prices.items() if k != "_meta"})
    p = repo_path(m["live_listings"])
    old = json.loads(p.read_text(encoding="utf-8")) if p.exists() else []
    p.write_text(
        json.dumps(old + [x.model_dump() for x in new], indent=1, ensure_ascii=False),
        encoding="utf-8",
    )
    summary["added"] = len(new)
    return summary

## Graph nodes

In [ ]:
#| export
def make_searching_node(
    data: MarketData | None = None,
    config: dict[str, Any] | None = None,
    live: bool | None = None,
    llm: LLM | None = None,
    scraper: MarketplaceScraper | None = None,
    pricebook: PriceBook | None = None,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the SEARCHING node. Coming back from EVALUATING counts as the one retry and
    relaxes the filters. With live search on, the first pass also scrapes Marketplace."""
    cfg = config or load_config()
    pool = data or load_data(cfg)
    use_live = cfg["merchant"]["live"] if live is None else live

    def searching_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        q = state.get("parsed_query") or ParsedQuery()
        retries = int(state.get("search_retry_count", 0))
        relaxed = bool(state.get("relaxed_filters", False))
        if state.get("current_state") == "EVALUATING":
            retries += 1
            relaxed = True

        update: dict[str, Any] = {}
        if use_live and not relaxed and q.card_name:
            model = llm or make_llm(cfg)
            book = pricebook or PriceBook(cfg, pool.prices)
            update["live_search"] = live_refresh(
                pool, q, cfg, model, scraper or MarketplaceScraper(cfg), book
            )

        candidates = find(pool.listings, q, cfg, relaxed=relaxed)
        timings = dict(state.get("stage_timings", {}))
        timings["SEARCHING"] = round(time.perf_counter() - t0, 4)
        return {
            **update,
            "current_state": "SEARCHING",
            "search_retry_count": retries,
            "relaxed_filters": relaxed,
            "candidate_listings": candidates,
            "stage_timings": timings,
        }

    return searching_node

In [ ]:
#| export
def evaluate_listing(listing: CardListing, data: MarketData, config: dict[str, Any]):
    """Risk check for one listing: (EvaluatedListing, RiskResult)."""
    risk = assess(listing, data.prices, data.red_flags, config)
    item = EvaluatedListing(
        listing=listing, market_price=risk.market, risk_score=risk.risk, risk_reasons=risk.signals
    )
    return item, risk

In [ ]:
#| export
def make_evaluating_node(
    data: MarketData | None = None,
    config: dict[str, Any] | None = None,
    llm: LLM | None = None,
    index: RedFlagIndex | None = None,
    reflect: bool = True,
) -> Callable[[AgentState], dict[str, Any]]:
    """Build the EVALUATING node: risk score for every candidate, a one-sentence reason for
    each flagged one, and the market facts for a consultative price question.

    reflect=False is ablation 3: no price check and no red-flag retrieval, every risk is 0.
    """
    cfg = config or load_config()
    pool = data or load_data(cfg)
    model = llm or make_llm(cfg)
    rag = index or (RedFlagIndex(pool.red_flags, cfg) if model.provider != "offline" else None)

    def evaluating_node(state: AgentState) -> dict[str, Any]:
        t0 = time.perf_counter()
        q = state.get("parsed_query") or ParsedQuery()
        evaluated: list[EvaluatedListing] = []
        flagged: list[tuple[CardListing, RiskResult]] = []
        for listing in state.get("candidate_listings", []):
            if not reflect:
                evaluated.append(EvaluatedListing(listing=listing))
                continue
            item, risk = evaluate_listing(listing, pool, cfg)
            evaluated.append(item)
            if risk.decision != "ok":
                flagged.append((listing, risk))

        reasons = explain_flags(model, flagged, rag)
        evaluated = [
            x.model_copy(update={"warning": reasons[x.listing.listing_id]})
            if x.listing.listing_id in reasons else x
            for x in evaluated
        ]  # fmt: skip

        timings = dict(state.get("stage_timings", {}))
        timings["EVALUATING"] = round(time.perf_counter() - t0, 4)
        return {
            "current_state": "EVALUATING",
            "evaluated_listings": evaluated,
            "reflection_complete": True,
            "price_check": price_check_facts(q, pool, cfg) if q.is_consultative else None,
            "stage_timings": timings,
        }

    return evaluating_node

## Offline stand-in for the explanation call

In [ ]:
#| export
@offline_handler("flag_reasons")
def offline_flag_reasons(_system: str, prompt: str) -> dict:
    """Rule-based stand-in for the flag-explanation call (offline mode)."""
    reasons = []
    for it in json.loads(between(prompt, "items")):
        sig = "; ".join(it.get("signals", [])[:2]) or "this listing has warning signs"
        reasons.append({"listing_id": it["listing_id"], "reason": sig[0].upper() + sig[1:] + "."})
    return {"reasons": reasons}

# Try it

In [ ]:
import pandas as pd

from src.config import load_config
from src.llm import make_llm
from src.pipeline import load_data

cfg = load_config()
llm = make_llm(cfg)
data = load_data(cfg)
print(
    f"{len(data.listings)} listings ({data.source}), {len(data.prices)} prices, {len(data.red_flags)} red flags"
)

In [ ]:
q = ParsedQuery(card_name="Charizard", set_code="Base Set", budget=400)
state = {"parsed_query": q, "current_state": "INTENT"}
state |= make_searching_node(data, cfg, live=False)(state)
state |= make_evaluating_node(data, cfg, llm=llm)(state)
pd.DataFrame(
    [
        {
            "id": x.listing.listing_id,
            "label": x.listing.label,
            "title": x.listing.raw_title[:38],
            "price": x.listing.price,
            "market": x.market_price,
            "risk": x.risk_score,
            "warning": x.warning,
        }
        for x in state["evaluated_listings"]
    ]
)

In [ ]:
pc = ParsedQuery(
    card_name="Mewtwo",
    set_code="Base Set",
    is_consultative=True,
    asked_price=45,
    min_condition="LP",
)
price_check_facts(pc, data, cfg)

Live search (costs Apify credit, needs `LIVE = True` for card identification). It adds new
listings to the pool and to `data/live_listings.json`.

In [ ]:
LIVE_SEARCH = False
if LIVE_SEARCH:
    live_data = load_data({**cfg, "data": {**cfg["data"], "use_samples": False}})
    summary = live_refresh(
        live_data,
        ParsedQuery(card_name="Charizard", set_code="Base Set"),
        cfg,
        llm,
        MarketplaceScraper(cfg),
        PriceBook(cfg, live_data.prices),
        log=print,
    )
    print(summary)

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()